In [1]:
%pip install sentence_transformers


  Using cached sentence_transformers-5.2.3-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)
  Using cached torch-2.10.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached typer_slim-0.24.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.0-py3-none-any.whl.metadata (16 kB)
  Using cached rich-14.3.2-py3-none-any.whl.metadata (18 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
Using cached sentence_transformers-5.2.3-py3-none-any.whl (494 kB)
Using cached huggingface_hub-1.4.1-py3-none-any.whl (553 kB)
Using cached torch-2.10.0-cp312-cp312-win_amd64.whl (113.8 MB)
Using cached transformers-5.2.0-py3-none-any.whl (10.4 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\nirma\\Training\\GenAI_Projects\\.venv\\Lib\\site-packages\\transformers\\models\\mm_grounding_dino\\modeling_mm_grounding_dino.py'
Check the permissions.


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Step 1: Define Sample Documents
documents = [
    {"section": "Employee Info", "content": "John's pay is processed on the 1st of every month."},
    {"section": "Employee Info", "content": "Mark is on a leave of absence until next Monday."},
    {"section": "Employee Info", "content": "Julie is a software engineer."},
    {"section": "Employee Info", "content": "Julie's pay is processed on the 1st of every month."},
    {"section": "Employee Info", "content": "Mark is a product manager."},
    {"section": "Employee Info", "content": "John is an AI architect and has salary of 500K USD."},
]

# Step 2: Get Content Texts
content_corpus = [doc["content"] for doc in documents]
content_corpus

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
doc_vectors = model.encode(content_corpus)

doc_vectors
print(doc_vectors.shape)


c:\Users\nirma\Training\GenAI_Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'sklearn.utils'

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPEN_AI_API_KEY")

my_client = OpenAI(api_key=my_api_key)
# my_client


# Define your target function that performs retrieval per-question
def ask_question_open_ai(prompt, context=""):
    """Call the LLM with the provided prompt and context.

    IMPORTANT: use the passed-in prompt (not a global variable) so each
    evaluation example can be answered correctly.
    """
    llm_response = my_client.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": '''
             You are an AI assistant who answers only based on the given context.
             '''},
            {"role": "user", "content": f"Context: {context}\n\nUser Question: {prompt}"}
        ]

    )
    print (llm_response)
    return llm_response.choices[0].message.content
    

In [ ]:
ask_question_open_ai("When is Summer solstice in 2026?")


In [ ]:
from langsmith import traceable

@traceable
def ask_question(inputs):
    question = inputs["question"]

    # Embed question
    query_vec = model.encode([question])[0]

    import numpy as np
    similarities = model.similarity(query_vec, doc_vectors)
    similarities = np.asarray(similarities).squeeze()

    # Top 3 retrieval
    top_3_indices = np.argsort(similarities)[::-1][:3]
    top_docs = [content_corpus[i] for i in top_3_indices]

    context = "\n---\n".join(top_docs)

    # Call LLM
    answer = ask_question_open_ai(question, context)

    return {
        "answer": answer,
        "contexts": top_docs  # must be list[str]
    }
    

In [ ]:
# %pip install --upgrade langsmith


In [ ]:

import langsmith
print(langsmith.__version__)


In [ ]:

reference_data = [
    {
        "inputs": {"question": "When is John's pay processed?"},
        "outputs": {"answer": "John's pay is processed on the 1st of every month."}
    },
    {
        "inputs": {"question": "What is Julie's job title?"},
        "outputs": {"answer": "Julie is a software engineer."}
    },
    {
        "inputs": {"question": "What is John's salary?"},
        "outputs": {"answer": "John has a salary of 500K USD."}
    },
    {
        "inputs": {"question": "What is Mark's current work status?"},
        "outputs": {"answer": "Mark is on a leave of absence until next Monday."}
    },
]


In [ ]:

def evaluate_context_recall(contexts, reference_answer):
    #context: John
    context_text = " ".join(contexts)
    return int(reference_answer.lower() in context_text.lower())


def evaluate_context_precision(contexts, reference_answer):
    relevant = sum(reference_answer.lower() in c.lower() for c in contexts)
    return relevant / len(contexts)


def evaluate_faithfulness(answer, contexts):
    context_text = " ".join(contexts)
    return int(answer.lower() in context_text.lower())


def evaluate_answer_correctness(answer, reference_answer):
    return int(answer.lower().strip() == reference_answer.lower().strip())
for data in reference_data:
    question = data["inputs"]["question"]
    reference = data["outputs"]["answer"]

    result = ask_question({"question": question})
    answer = result["answer"]
    contexts = result["contexts"]

    recall = evaluate_context_recall(contexts, reference)
    precision = evaluate_context_precision(contexts, reference)
    faith = evaluate_faithfulness(answer, contexts)
    correctness = evaluate_answer_correctness(answer, reference)

    print("\n==============================")
    print("Question:", question)
    print("Answer:", answer)
    print("Reference:", reference)

    print("\nMetrics:")
    print("Context Recall:", recall)
    print("Context Precision:", round(precision, 2))
    print("Faithfulness:", faith)
    print("Answer Correctness:", correctness)